# Tutorial: Reproducing the Analog Accuracy Gap on Tiny-ViT

**Goal:** In ≤30 minutes, reproduce the dramatic accuracy collapse (86.37% → ~10%) that occurs when a Tiny-ViT vision transformer is deployed on noisy analog hardware without proper hardware-aware training (HAT).

**Audience:** ML engineers who have never thought about analog hardware.

**What you will learn:**
1. How to load a hybrid analog/digital Tiny-ViT checkpoint.
2. How device-to-device (D2D) noise creates a "fresh instance" mismatch.
3. Why *Ensemble HAT* rescues accuracy by resampling D2D mismatch during training.
4. How to visualize the gap in a single matplotlib bar chart.

> **Prerequisites:** PyTorch, torchvision, `timm`, and the `compute_vit` repository on your `PYTHONPATH`.


## 1. Setup — Imports, device, and reproducibility

We need `torch` and the `compute_vit` training/evaluation modules.
Because this notebook lives in `notebooks/`, we append the parent directory so Python can find the repo modules.


In [ ]:
import sys
from pathlib import Path

# ---------------------------------------------------------------
# Add repo root to PYTHONPATH
# ---------------------------------------------------------------
repo_root = Path(__file__).resolve().parents[1]  # notebooks/ → compute_vit/
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from statistics import mean, stdev

# compute_vit imports
try:
    import timm
except ImportError:
    raise ImportError("timm is required. Install: pip install timm")

from train_tinyvit import set_seed
from train_tinyvit_ensemble import (
    TinyViTExperimentConfig,
    build_model,
    evaluate,
    get_dataloaders,
    resample_all_d2d_noise,
)
from device_profile_utils import DeviceProfile

# ---------------------------------------------------------------
# Device & reproducibility
# ---------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

set_seed(42)


## 2. Load the Tiny-ViT V4 checkpoint

The checkpoint `V4_hybrid_standard_noise_hat_best.pt` is the **Ensemble HAT** model used in the paper.
It was trained with:

* **Hybrid analog/digital layers** (linear & conv layers mapped to conductance arrays).
* **Standard organic noise** (σ_D2D = 10%, σ_C2C = 5%).
* **Ensemble HAT:** every training epoch starts on a *fresh* D2D mismatch instance.

Because the checkpoint also stores the `TinyViTExperimentConfig` used during training, we can reconstruct the exact model architecture and noise settings automatically.


In [ ]:
import dataclasses

# ---------------------------------------------------------------
# Path to the paper-locked V4 checkpoint (relative to repo root)
# ---------------------------------------------------------------
CKPT_PATH = "../checkpoints/V4_hybrid_standard_noise_hat_best.pt"

# Load checkpoint metadata and weights
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

# Reconstruct the experiment config from the checkpoint payload
exp_cfg_dict = ckpt.get("exp_cfg", {})
valid_keys = {f.name for f in dataclasses.fields(TinyViTExperimentConfig)}
filtered = {k: v for k, v in exp_cfg_dict.items() if k in valid_keys}
cfg = TinyViTExperimentConfig(**filtered)

print(f"Checkpoint epoch: {ckpt.get('epoch', 'N/A')}")
print(f"Training best accuracy: {ckpt.get('best_acc', 'N/A'):.2f}%")
print(f"Experiment name: {cfg.name}")
print(f"  use_hybrid    = {cfg.use_hybrid}")
print(f"  noise_enabled = {cfg.noise_enabled}")
print(f"  sigma_c2c     = {cfg.sigma_c2c}")
print(f"  sigma_d2d     = {cfg.sigma_d2d}")
print(f"  hat_training  = {cfg.hat_training}")


## 3. Define an i.i.d. D2D device profile (σ = 10%)

A `DeviceProfile` describes the physical properties of the analog crossbar array.
For this walkthrough we use the **standard organic OPECT** profile (uniform D2D noise):

| Field | Value | Meaning |
|-------|-------|---------|
| `device_type` | `"Organic OPECT"` | Human-readable label |
| `dynamic_range` | `10.0` | G_max / G_min conductance ratio |
| `n_states` | `16` | 4-bit discrete conductance levels |
| `sigma_c2c` | `0.05` | Cycle-to-cycle noise (5%) |
| `sigma_d2d` | `0.10` | Device-to-device noise (10%) |
| `noise_mode` | `"uniform"` | Noise magnitude is σ × G_range |
| `source` | `"tutorial"` | Citation / provenance |

> **Note:** The profile is *frozen* (`frozen=True` dataclass) so it can be hashed and used as a lookup key.


In [ ]:
# ---------------------------------------------------------------
# Create a standard organic device profile
# ---------------------------------------------------------------
profile = DeviceProfile(
    device_type="Organic OPECT",
    dynamic_range=10.0,   # G_max = 10 × G_min
    n_states=16,          # 4-bit quantization
    sigma_c2c=0.05,       # 5% cycle-to-cycle noise
    sigma_d2d=0.10,       # 10% device-to-device noise
    noise_mode="uniform", # uniform noise scaling
    source="tutorial / Guo 2024 §2.1",
)

print("DeviceProfile created:")
print(f"  type       = {profile.device_type}")
print(f"  G_min      = {profile.G_min}")
print(f"  G_max      = {profile.G_max}")
print(f"  n_states   = {profile.n_states}")
print(f"  σ_C2C      = {profile.sigma_c2c}")
print(f"  σ_D2D      = {profile.sigma_d2d}")
print(f"  noise_mode = {profile.noise_mode}")


## 4. Fresh-instance evaluation — naive fixed-mask HAT

Here is the **mistake** most ML engineers make when moving to analog:

1. Train a model with HAT.
2. At inference time, load the checkpoint and evaluate.
3. Never touch the D2D mismatch buffers again.

Because the D2D noise is *fixed* to whatever instance happened to be sampled at the end of training, the model sees a **permanent train/test mismatch**.
On 10 fresh evaluation seeds (5 Monte-Carlo passes each) the accuracy collapses to roughly **chance level (~10%)**.


In [ ]:
# ---------------------------------------------------------------
# Helper: evaluate WITHOUT resampling D2D (fixed-mask / naive)
# ---------------------------------------------------------------
@torch.no_grad()
def evaluate_fixed_mask(
    checkpoint: dict,
    exp_cfg: TinyViTExperimentConfig,
    device: str,
    fresh_instances: int = 10,
    eval_runs: int = 5,
    data_root: str = "../data",
    num_workers: int = 0,
):
    """
    Simulate naive deployment: the D2D mismatch baked into the checkpoint
    stays frozen forever.  We only vary the random seed (which affects C2C
    noise sampling during forward passes), but the underlying device instance
    never changes.
    """
    _, testloader = get_dataloaders(
        dataset="cifar10",
        batch_size=256,
        data_root=data_root,
        num_workers=num_workers,
    )
    criterion = nn.CrossEntropyLoss()

    instance_accs = []
    for i in range(fresh_instances):
        seed = 42 + i * 100
        set_seed(seed)

        # Re-build model and load checkpoint weights
        model = build_model(exp_cfg, num_classes=10, device=device, pretrained=False)
        model.load_state_dict(checkpoint["model_state_dict"], strict=True)

        # NAIVE: we do NOT call resample_all_d2d_noise(model)
        #        → D2D stays locked to the final training instance

        run_accs = []
        for _ in range(eval_runs):
            _, acc = evaluate(
                model, testloader, criterion, device, exp_cfg, amp_enabled=False
            )
            run_accs.append(float(acc))

        instance_mean = mean(run_accs)
        instance_accs.append(instance_mean)
        print(f"  Instance {i+1:02d}/{fresh_instances}: mean_acc={instance_mean:.2f}%")

    return instance_accs


In [ ]:
# Run the naive evaluation (this may take a few minutes)
print("=" * 60)
print("Fixed-mask HAT — NO D2D resampling")
print("=" * 60)
fixed_mask_accs = evaluate_fixed_mask(ckpt, cfg, device, fresh_instances=10, eval_runs=5)

print(f"\nFixed-mask mean : {mean(fixed_mask_accs):.2f}%")
if len(fixed_mask_accs) > 1:
    print(f"Fixed-mask std  : {stdev(fixed_mask_accs):.2f}%")


## 5. Ensemble HAT — resample D2D for every fresh instance

The *same* checkpoint is now evaluated correctly.
Before each fresh instance we call `resample_all_d2d_noise(model)`, which draws a **new D2D mismatch pattern** for every analog layer.
This matches the training distribution (Ensemble HAT resampled D2D at every epoch), so the model generalizes again.

**Why does this work?**
During Ensemble HAT training, the model learns a *population* of weights that is robust to the average D2D mismatch across many device instances.
By resampling D2D at test time, each inference draws from that same population, and the average accuracy recovers to ~86%.


In [ ]:
# ---------------------------------------------------------------
# Helper: evaluate WITH D2D resampling (Ensemble HAT)
# ---------------------------------------------------------------
@torch.no_grad()
def evaluate_ensemble_hat(
    checkpoint: dict,
    exp_cfg: TinyViTExperimentConfig,
    device: str,
    fresh_instances: int = 10,
    eval_runs: int = 5,
    data_root: str = "../data",
    num_workers: int = 0,
):
    """
    Correct deployment of an Ensemble HAT checkpoint:
    every fresh device instance gets its own D2D mismatch draw.
    """
    _, testloader = get_dataloaders(
        dataset="cifar10",
        batch_size=256,
        data_root=data_root,
        num_workers=num_workers,
    )
    criterion = nn.CrossEntropyLoss()

    instance_accs = []
    for i in range(fresh_instances):
        seed = 42 + i * 100
        set_seed(seed)

        model = build_model(exp_cfg, num_classes=10, device=device, pretrained=False)
        model.load_state_dict(checkpoint["model_state_dict"], strict=True)

        # KEY: resample D2D mismatch → new physical instance
        resampled_count = resample_all_d2d_noise(model)

        run_accs = []
        for _ in range(eval_runs):
            _, acc = evaluate(
                model, testloader, criterion, device, exp_cfg, amp_enabled=False
            )
            run_accs.append(float(acc))

        instance_mean = mean(run_accs)
        instance_accs.append(instance_mean)
        print(f"  Instance {i+1:02d}/{fresh_instances}: mean_acc={instance_mean:.2f}%  "
              f"(resampled {resampled_count} layers)")

    return instance_accs


# Run the Ensemble HAT evaluation
print("=" * 60)
print("Ensemble HAT — resample D2D per instance")
print("=" * 60)
ensemble_hat_accs = evaluate_ensemble_hat(ckpt, cfg, device, fresh_instances=10, eval_runs=5)

print(f"\nEnsemble HAT mean : {mean(ensemble_hat_accs):.2f}%")
if len(ensemble_hat_accs) > 1:
    print(f"Ensemble HAT std  : {stdev(ensemble_hat_accs):.2f}%")


## 6. Plot the ranking gap

A single bar chart is worth a thousand words.
We compare the mean fresh-instance accuracy ± standard deviation for:

* **Fixed-mask HAT** — D2D locked to the final training instance.
* **Ensemble HAT** — D2D resampled per instance.


In [ ]:
# ---------------------------------------------------------------
# Bar chart: Fixed-mask vs Ensemble HAT
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 4.5))

means = [mean(fixed_mask_accs), mean(ensemble_hat_accs)]
stds = [
    stdev(fixed_mask_accs) if len(fixed_mask_accs) > 1 else 0.0,
    stdev(ensemble_hat_accs) if len(ensemble_hat_accs) > 1 else 0.0,
]
labels = ["Fixed-mask HAT\n(naive)", "Ensemble HAT\n(resample D2D)"]
colors = ["#e74c3c", "#2ecc71"]  # red = bad, green = good

x_pos = [0, 1]
bars = ax.bar(
    x_pos, means, yerr=stds, capsize=6, color=colors, width=0.5,
    edgecolor="black", linewidth=0.8,
)

ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("CIFAR-10 Test Accuracy (%)", fontsize=11)
ax.set_title("Fresh-Instance Accuracy: Fixed-mask vs Ensemble HAT", fontsize=12, fontweight="bold")
ax.set_ylim(0, 100)
ax.axhline(y=10.0, color="gray", linestyle="--", linewidth=1.0, label="Chance level (10 classes)")
ax.legend(loc="upper right")

# Annotate bars with numeric values
for bar, m, s in zip(bars, means, stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + s + 3,
        f"{m:.2f}%",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )

plt.tight_layout()
plt.show()

print(f"\nGap recovered: {means[1] - means[0]:.2f} percentage points")


## 6.5: The severe-NL ceiling

What happens when write nonlinearity becomes severe (NL=2.0)?

### 6.5.1 MLP-only checkpoint at NL=2.0

We trained a model where **only the MLP blocks** are mapped to analog arrays, while the attention layers remain in digital precision.  This removes the analog noise burden from the most sensitive part of the transformer, so we expect it to be more robust.  Even so, at severe nonlinearity (NL=2.0) the MLP-only hybrid hits a hard ceiling.

> **Checkpoint:** `V4_mlp_only_nl20_best.pt`

In [ ]:
# ---------------------------------------------------------------
# Load MLP-only checkpoint (severe NL=2.0) and evaluate on fresh instances
# ---------------------------------------------------------------
CKPT_MLP_ONLY = "../checkpoints/V4_mlp_only_nl20_best.pt"

ckpt_mlp = torch.load(CKPT_MLP_ONLY, map_location="cpu", weights_only=False)
exp_cfg_mlp = TinyViTExperimentConfig(**{k: v for k, v in ckpt_mlp.get("exp_cfg", {}).items()
                                        if k in valid_keys})

print("MLP-only checkpoint:")
print(f"  Training best accuracy: {ckpt_mlp.get('best_acc', 'N/A'):.2f}%")
print(f"  write_nonlinearity = {getattr(exp_cfg_mlp, 'write_nonlinearity', 'N/A')}")

# Evaluate on fresh device instances (same protocol as Section 5)
mlp_accs = evaluate_ensemble_hat(
    ckpt_mlp, exp_cfg_mlp, device,
    fresh_instances=10, eval_runs=5
)
print(f"\nMLP-only (NL=2.0) mean : {mean(mlp_accs):.2f}%")
print(f"MLP-only (NL=2.0) std  : {stdev(mlp_accs):.2f}%")


### 6.5.2 Joint-training checkpoint at NL=2.0

Next we evaluate a **joint-training** checkpoint: analog MLP + analog attention trained together with HAT and D2D resampling.  Joint training is the most aggressive mitigation because the attention layers can adapt their representations to compensate for MLP distortion.  Yet the ceiling is essentially the same.

> **Checkpoint:** `V4_joint_nl20_best.pt`

In [ ]:
# ---------------------------------------------------------------
# Load joint-training checkpoint (severe NL=2.0) and evaluate on fresh instances
# ---------------------------------------------------------------
CKPT_JOINT = "../checkpoints/V4_joint_nl20_best.pt"

ckpt_joint = torch.load(CKPT_JOINT, map_location="cpu", weights_only=False)
exp_cfg_joint = TinyViTExperimentConfig(**{k: v for k, v in ckpt_joint.get("exp_cfg", {}).items()
                                           if k in valid_keys})

print("Joint-training checkpoint:")
print(f"  Training best accuracy: {ckpt_joint.get('best_acc', 'N/A'):.2f}%")
print(f"  write_nonlinearity = {getattr(exp_cfg_joint, 'write_nonlinearity', 'N/A')}")

joint_accs = evaluate_ensemble_hat(
    ckpt_joint, exp_cfg_joint, device,
    fresh_instances=10, eval_runs=5
)
print(f"\nJoint (NL=2.0) mean : {mean(joint_accs):.2f}%")
print(f"Joint (NL=2.0) std  : {stdev(joint_accs):.2f}%")


### 6.5.3 Interpretation

Three independent mitigations all hit the same ceiling. This suggests a structural limit in the attention pathway, not a training artifact.

### 6.5.4 Visual comparison: the severe-NL ceiling

The bar chart below overlays the two severe-NL results on the Ensemble HAT baseline (NL=1.0).  The gap is striking: even the best-trained analog model at NL=2.0 recovers only about a third of the accuracy achieved at mild nonlinearity.

In [ ]:
# ---------------------------------------------------------------
# Bar chart: Ensemble HAT (NL=1.0) vs severe-NL mitigations (NL=2.0)
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4.5))

means = [mean(ensemble_hat_accs), mean(mlp_accs), mean(joint_accs)]
stds = [
    stdev(ensemble_hat_accs) if len(ensemble_hat_accs) > 1 else 0.0,
    stdev(mlp_accs) if len(mlp_accs) > 1 else 0.0,
    stdev(joint_accs) if len(joint_accs) > 1 else 0.0,
]
labels = ["Ensemble HAT\n(NL=1.0)", "MLP-only\n(NL=2.0)", "Joint\n(NL=2.0)"]
colors = ["#2ecc71", "#f39c12", "#e74c3c"]  # green, orange, red

x_pos = [0, 1, 2]
bars = ax.bar(
    x_pos, means, yerr=stds, capsize=6, color=colors, width=0.5,
    edgecolor="black", linewidth=0.8,
)

ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("CIFAR-10 Test Accuracy (%)", fontsize=11)
ax.set_title("The Severe-NL Ceiling: Ensemble HAT vs NL=2.0 Mitigations", fontsize=12, fontweight="bold")
ax.set_ylim(0, 100)
ax.axhline(y=10.0, color="gray", linestyle="--", linewidth=1.0, label="Chance level (10 classes)")
ax.legend(loc="upper right")

for bar, m, s in zip(bars, means, stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + s + 3,
        f"{m:.2f}%",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )

plt.tight_layout()
plt.show()

print(f"\nGap NL=1.0 → MLP-only NL=2.0 : {means[0] - means[1]:.2f} pp")
print(f"Gap NL=1.0 → Joint NL=2.0      : {means[0] - means[2]:.2f} pp")


## 7. Next steps & further reading

### Try different device profiles
Swap the `DeviceProfile` in Section 3 to explore how other technologies behave:

```python
# RRAM (HfOx) — lower noise, higher dynamic range
rram = DeviceProfile(
    device_type="RRAM (HfOx)",
    dynamic_range=100.0,
    n_states=64,
    sigma_c2c=0.02,
    sigma_d2d=0.05,
    source="Alibart 2016; Prezioso 2015",
)
```

### Thesis chapters
* **Chapter 3** — Conductance quantization & noise models.
* **Chapter 4** — Hardware-Aware Training (HAT) and the Ensemble HAT algorithm.
* **Chapter 5** — Full cross-dataset evaluation (CIFAR-10, CIFAR-100, Flowers-102).

### Code repositories
* `compute_vit` — This repo: Tiny-ViT training, evaluation, and profiling.
* `cross-sim` — CrossSim analog accelerator simulator (used for energy estimates).

### Relevant scripts in this repo
| Script | Purpose |
|--------|---------|
| `run_ensemble_hat_fixed.py` | One-shot sanity check of the Ensemble HAT gap |
| `run_fresh_instance_cadence_control.py` | Full 10×5 fresh-instance protocol used for the paper |
| `eval_literature_profile.py` | Evaluate against literature device profiles |
| `run_visualization_suite.py` | Generate attention maps and layer-sensitivity plots |

> **Tip:** If you want to train your own Ensemble HAT model from scratch, start with `train_tinyvit_ensemble.py --experiment V4`.
